# Kid Drawing Story App in Colab

This notebook installs the project, checks for a GPU runtime, and launches the Gradio app. Upload or clone the full repo first so the `story_app/` package is available in Colab.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = os.environ.get("KID_STORY_REPO_URL", "")
CANDIDATES = [
    Path.cwd(),
    Path("/content/kid-drawing-story-app"),
    Path("/content/New project"),
    Path("/content/drive/MyDrive/kid-drawing-story-app"),
]

repo_dir = next((path for path in CANDIDATES if (path / "pyproject.toml").exists()), None)
if repo_dir is None and REPO_URL:
    repo_dir = Path("/content/kid-drawing-story-app")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)

if repo_dir is None or not (repo_dir / "pyproject.toml").exists():
    raise RuntimeError(
        "Could not find the full project files. Upload or clone the repo, or set KID_STORY_REPO_URL before running this cell."
    )

os.chdir(repo_dir)
print(f"Using repo: {repo_dir}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "espeak-ng"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, switch Runtime > Change runtime type > GPU and rerun.")

print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from story_app.app import build_demo

demo = build_demo()
demo.launch(debug=True, share=True, inline=False)
